In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print("Ready")

Ready


In [2]:
%%writefile agents/report_generator_agent.py
from typing import Dict, List
from collections import Counter

from tools.llm_clients import call_llm
from tools.logger import log_agent_event

AGENT_NAME = "report_generator_agent"


def _build_narrative_prompt(query: str, summaries: List[Dict]) -> str:
    sections = "\n\n".join(
        f"## {s['sub_question']}\n{s['summary']}" for s in summaries
    )
    return f"""Write a clear, well-organized research report answering this question:
"{query}"

Use the sub-topic briefs below as your source material. Combine them into flowing,
coherent prose organized under clear headings — do not just copy them verbatim, but
do not invent any facts, numbers, or claims not present in the briefs either.
Preserve any hedging language (e.g. "according to a single source", "this is disputed")
exactly as given — do not remove it or make claims sound more certain than the briefs do.

Sub-topic briefs:
{sections}

Write the report in Markdown with a short intro, one section per sub-topic, and a
one-paragraph conclusion. Do not include a confidence/sources appendix — that will
be added separately.
"""


def _generate_narrative(query: str, summaries: List[Dict]) -> Dict:
    """Generates the narrative body via LLM. On failure, falls back to a plain
    concatenation of the summaries — never returns nothing, since this is the
    final user-facing output."""
    try:
        text = call_llm(_build_narrative_prompt(query, summaries)).strip()
        return {"text": text, "method": "llm"}
    except Exception as e:
        log_agent_event(AGENT_NAME, "narrative_failed", error=f"{type(e).__name__}: {e}")
        fallback = f"# Research Report: {query}\n\n" + "\n\n".join(
            f"## {s['sub_question']}\n{s['summary']}" for s in summaries
        )
        return {"text": fallback, "method": "fallback_template"}


def _build_confidence_appendix(fact_check_results: List[Dict]) -> str:
    """Pure Python, no LLM — cannot fail or hallucinate. Deterministically
    summarizes confidence levels and lists every disputed/unverified claim."""
    if not fact_check_results:
        return "\n\n---\n## Confidence Appendix\nNo fact-checked claims were available.\n"

    counts = Counter(f["status"] for f in fact_check_results)
    total = len(fact_check_results)

    lines = ["\n\n---\n## Confidence Appendix\n"]
    lines.append(f"**Total claims reviewed:** {total}  ")
    lines.append(f"**Confirmed:** {counts.get('CONFIRMED', 0)} | "
                f"**Disputed:** {counts.get('DISPUTED', 0)} | "
                f"**Unverified:** {counts.get('UNVERIFIED', 0)}\n")

    disputed = [f for f in fact_check_results if f["status"] == "DISPUTED"]
    if disputed:
        lines.append("###  Disputed claims\n")
        for f in disputed:
            lines.append(f"- \"{f['claim']}\" — *{f.get('reasoning', '')}*")

    unverified = [f for f in fact_check_results if f["status"] == "UNVERIFIED"]
    if unverified:
        lines.append("\n###  Unverified claims (single source, no corroboration)\n")
        for f in unverified:
            lines.append(f"- \"{f['claim']}\"")

    return "\n".join(lines)


def generate_report(query: str, summaries: List[Dict], fact_check_results: List[Dict]) -> Dict:
    """Assembles the full report. Guaranteed to return a non-empty report even
    if the LLM narrative generation fails, since the appendix is LLM-free."""
    if not summaries:
        return {
            "report": f"# Research Report: {query}\n\nNo verified information was found for this query.",
            "method": "no_data",
        }

    narrative = _generate_narrative(query, summaries)
    appendix = _build_confidence_appendix(fact_check_results)
    full_report = narrative["text"] + appendix

    log_agent_event(AGENT_NAME, "report_done", method=narrative["method"],
                    sub_topics=len(summaries), total_claims=len(fact_check_results))

    return {"report": full_report, "method": narrative["method"]}


def report_generator_node(state: Dict) -> Dict:
    """LangGraph node: assembles the final report from state."""
    summaries = state.get("summaries", [])
    fact_check_results = state.get("fact_check_results", [])
    query = state.get("query", "Unknown query")

    if not summaries:
        return {"errors": [{"agent": AGENT_NAME, "error": "no summaries to build report from"}]}

    result = generate_report(query, summaries, fact_check_results)

    return {
        "final_report": result["report"],
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "method": result["method"]}],
    }

Writing agents/report_generator_agent.py


In [3]:
from agents.web_search_agent import search_one
from agents.data_extraction_agent import extract_facts_from_result
from agents.fact_checking_agent import fact_check_facts
from agents.summarization_agent import summarize_sub_topic
from agents.report_generator_agent import generate_report

search_result = search_one("What are the main safety risks of solid-state EV batteries?")
facts = extract_facts_from_result(search_result)
checked = fact_check_facts(facts)
summary = summarize_sub_topic(search_result["sub_question"], checked)

result = generate_report(
    query="What are the main safety risks of solid-state EV batteries?",
    summaries=[summary],
    fact_check_results=checked,
)

print("Method:", result["method"])
print("\n" + result["report"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


 Gemini unavailable (ClientError), falling back to Tavily...


2026-10-04 16:59:07,042 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3
2026-10-04 16:59:09,717 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the main safety risks of solid-state EV batteries? facts_found=14
2026-10-04 16:59:10,958 | INFO | research_system | [fact_checking_agent] batch_done CONFIRMED=13 UNVERIFIED=1
2026-10-04 16:59:14,381 | INFO | research_system | [summarization_agent] summary_done sub_question=What are the main safety risks of solid-state EV batteries? status=ok CONFIRMED=13 DISPUTED=0 UNVERIFIED=1
2026-10-04 16:59:23,515 | INFO | research_system | [report_generator_agent] report_done method=llm sub_topics=1 total_claims=14


Method: llm

# Safety Risks of Solid-State EV Batteries

## Introduction
The transition from traditional liquid electrolytes to solid-state batteries represents a significant shift in electric vehicle (EV) battery technology. While this innovation promises enhanced safety profiles by addressing several well-known hazards of conventional systems, it is not without its own set of challenges. This report examines the primary safety risks associated with solid-state EV batteries, balancing the confirmed benefits of the new design against the residual dangers and performance limitations that remain.

## Mitigation of Traditional Hazards
Solid-state batteries are designed to mitigate several critical safety risks inherent in traditional liquid electrolyte systems. A key advantage of this technology is that the solid electrolyte is confirmed to be thermally stable and non-combustible. This fundamental material property significantly reduces the risk of thermal runaway, a dangerous chain react

In [4]:
from agents.report_generator_agent import generate_report

disputed_facts = [
    {"claim": "Solid-state batteries have a lower fire risk than liquid-electrolyte batteries.",
     "status": "DISPUTED", "reasoning": "contradicts the next claim", "sources": []},
    {"claim": "Solid-state batteries have the same fire risk as liquid-electrolyte batteries.",
     "status": "DISPUTED", "reasoning": "contradicts the previous claim", "sources": []},
    {"claim": "Tesla's headquarters is in Austin, Texas.",
     "status": "UNVERIFIED", "reasoning": "single source", "sources": []},
]

from agents.summarization_agent import summarize_sub_topic
disputed_summary = summarize_sub_topic("What is the fire risk of solid-state batteries?", disputed_facts)

result = generate_report(
    query="Test query with disputes",
    summaries=[disputed_summary],
    fact_check_results=disputed_facts,
)
print(result["report"])

assert "Disputed claims" in result["report"], "FAIL: appendix should list disputed claims"
assert "Unverified claims" in result["report"], "FAIL: appendix should list unverified claims"
print("\n Confidence appendix correctly lists disputed and unverified claims")

2026-10-04 17:05:58,938 | INFO | research_system | [summarization_agent] summary_done sub_question=What is the fire risk of solid-state batteries? status=ok CONFIRMED=0 DISPUTED=2 UNVERIFIED=1
2026-10-04 17:05:59,531 | INFO | research_system | [report_generator_agent] report_done method=llm sub_topics=1 total_claims=3


# Research Report: Test Query with Disputes

## Introduction
This report addresses the specific inquiry regarding the fire risk associated with solid-state batteries. The analysis is based on available sub-topic briefs that outline the current state of knowledge and existing disagreements within the field. The following section details the specific findings and disputes identified in the source material.

## Fire Risk of Solid-State Batteries
Currently, there are no confirmed facts available regarding the specific fire risk of solid-state batteries. The relative safety profile of this technology is currently disputed. One side of the argument posits that solid-state batteries possess a lower fire risk compared to liquid-electrolyte batteries. Conversely, the other side contends that solid-state batteries carry the same fire risk as their liquid-electrolyte counterparts. At present, no other verified information is provided to resolve this disagreement.

## Conclusion
In summary, the sp

In [5]:
import agents.report_generator_agent as rga

original = rga.call_llm
rga.call_llm = lambda p: (_ for _ in ()).throw(RuntimeError("simulated outage"))

result = rga.generate_report("Test query", [summary], checked)
print("Method used:", result["method"])
assert result["method"] == "fallback_template"
assert len(result["report"]) > 0
assert "Confidence Appendix" in result["report"]
print("Fail-safe confirmed: report still complete even when LLM narrative fails")

rga.call_llm = original

2026-10-04 17:23:30,198 | INFO | research_system | [report_generator_agent] narrative_failed error=RuntimeError: simulated outage
2026-10-04 17:23:30,199 | INFO | research_system | [report_generator_agent] report_done method=fallback_template sub_topics=1 total_claims=14


Method used: fallback_template
Fail-safe confirmed: report still complete even when LLM narrative fails


In [6]:
from agents.report_generator_agent import report_generator_node

state = {
    "query": "What are the main safety risks of solid-state EV batteries?",
    "summaries": [summary],
    "fact_check_results": checked,
}
out = report_generator_node(state)
print(out["final_report"][:500])
print("\n...\n")
print("Model usage:", out["model_usage_log"])

2026-10-04 17:24:32,840 | INFO | research_system | [report_generator_agent] report_done method=llm sub_topics=1 total_claims=14


# Safety Profile of Solid-State EV Batteries

## Introduction
The transition from traditional liquid electrolytes to solid-state electrolytes in electric vehicle (EV) batteries represents a significant shift in safety engineering. While this technology is designed to address several critical hazards associated with conventional battery systems, it is not without its own set of challenges. This report outlines the primary safety risks and performance limitations of solid-state EV batteries, disti

...

Model usage: [{'agent': 'report_generator_agent', 'provider': 'groq', 'method': 'llm'}]


In [ ]:
%%writefile tests/test_report_generator_agent.py
import agents.report_generator_agent as rga


def test_generate_report_no_summaries():
    result = rga.generate_report("q", [], [])
    assert result["method"] == "no_data"
    assert "No verified information" in result["report"]


def test_generate_report_llm_success(monkeypatch):
    monkeypatch.setattr(rga, "call_llm", lambda p: "A polished narrative.")
    summaries = [{"sub_question": "q1", "summary": "brief 1"}]
    facts = [{"claim": "A", "status": "CONFIRMED", "reasoning": "", "sources": []}]
    result = rga.generate_report("q", summaries, facts)
    assert result["method"] == "llm"
    assert "A polished narrative." in result["report"]
    assert "Confidence Appendix" in result["report"]


def test_generate_report_llm_failure_falls_back(monkeypatch):
    def boom(p):
        raise RuntimeError("down")
    monkeypatch.setattr(rga, "call_llm", boom)
    summaries = [{"sub_question": "q1", "summary": "brief 1"}]
    result = rga.generate_report("q", summaries, [])
    assert result["method"] == "fallback_template"
    assert "brief 1" in result["report"]


def test_appendix_lists_disputed_and_unverified():
    facts = [
        {"claim": "Disputed one", "status": "DISPUTED", "reasoning": "x", "sources": []},
        {"claim": "Unverified one", "status": "UNVERIFIED", "reasoning": "y", "sources": []},
        {"claim": "Confirmed one", "status": "CONFIRMED", "reasoning": "z", "sources": []},
    ]
    appendix = rga._build_confidence_appendix(facts)
    assert "Disputed one" in appendix
    assert "Unverified one" in appendix
    assert "**Confirmed:** 1" in appendix  # matches the actual bold-markdown format produced


def test_appendix_empty_facts():
    appendix = rga._build_confidence_appendix([])
    assert "No fact-checked claims" in appendix


def test_node_empty_summaries():
    out = rga.report_generator_node({"query": "q", "summaries": [], "fact_check_results": []})
    assert out["errors"] and "final_report" not in out


def test_node_success(monkeypatch):
    monkeypatch.setattr(rga, "call_llm", lambda p: "narrative text")
    state = {
        "query": "q",
        "summaries": [{"sub_question": "q1", "summary": "brief"}],
        "fact_check_results": [{"claim": "A", "status": "CONFIRMED", "reasoning": "", "sources": []}],
    }
    out = rga.report_generator_node(state)
    assert "final_report" in out
    assert "narrative text" in out["final_report"]

Writing tests/test_report_generator_agent.py
